> 在线讲义：[第 8 章 · Dataset 与 DataLoader：给 GPU 备菜的流水线](https://wikieden.github.io/pytorch-inside-out/lessons/08-dataloader.html) · 本书《PyTorch 从里到外》

# 第 8 章 · Dataset 与 DataLoader：给 GPU 备菜的流水线

本章回答：`Dataset` 和 `DataLoader` 各管什么？一个 batch 是怎么拼出来的？`num_workers`、`pin_memory`、`collate_fn` 是干嘛的？为什么数据加载慢会让 GPU 闲着？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/08-dataloader.md`（讲解、动画、练习都在讲义里）。对应原文：00-overview.md §3、§6.2 第一行；官方文档 `torch.utils.data`（〔在线〕）；AISPE ch13「优化数据输入流水线」（`ch13.md` 第 1552–1561 行）；本章输出在本机 torch 2.15.0 上跑过（代码见本章末尾"代码对照"）

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 分批 | ⌈10 / 4⌉ = 3 | `batches()` | [4, 4, 2]；drop_last 时 2 批 |
| 第二步 补齐 | 补到最长 | `pad_demo()` | 长度 2、5、3 → 5 |
| 第三步 worker 数 | k ≥ t_d / t_s | `gpu_busy(40, 25, k)` | 0 个 38%，2 个 100% |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `batches`**（来自 `ch08_dataloader.py`）：下面要用到的小工具。

In [ ]:
def batches(n: int = 10, batch_size: int = 4, drop_last: bool = False):
    ds = TensorDataset(torch.randn(n, 4), torch.arange(n))
    return [(tuple(x.shape), y.tolist()) for x, y in DataLoader(ds, batch_size=batch_size, drop_last=drop_last)]

**函数 `gpu_busy`**（来自 `ch08_dataloader.py`）：稳定后 GPU 忙碌占比：0 个 worker 串行；k 个 worker 时供给速度 k / t_data。

In [ ]:
def gpu_busy(t_data: float, t_step: float, workers: int) -> float:
    """稳定后 GPU 忙碌占比：0 个 worker 串行；k 个 worker 时供给速度 k / t_data。"""
    if workers == 0:
        return t_step / (t_data + t_step)
    per_batch = t_data / workers
    return t_step / max(t_step, per_batch)

**函数 `pad_demo`**（来自 `ch08_dataloader.py`）：下面要用到的小工具。

In [ ]:
def pad_demo():
    seqs = [torch.ones(2), torch.ones(5), torch.ones(3)]
    return torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 10 个样本、每批 4：", batches())
print("drop_last=True 时批数：", len(batches(drop_last=True)))

In [ ]:
print("\n第二步 补齐：\n", pad_demo())

In [ ]:
print("\n第三步 读一批 40 ms、训一步 25 ms 时 GPU 忙碌占比：")
for k in (0, 1, 2, 4):
    print(f"  num_workers={k}: {gpu_busy(40, 25, k):.0%}")

## 3. 代码里才看得到的

1. **`gpu_busy` 是理想模型**：只算稳定后的吞吐，不算启动 worker 的开销和第一批的等待。
2. **`pad_sequence` 默认补 0**：要补别的值用 `padding_value=`。

## 4. 自己改一改

- `gpu_busy(60, 25, k)` 对 k = 1、2、3 各算一遍，看要几个 worker。
- 给 `pad_demo` 加 `padding_value=-1`。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> Dataset 管第 i 个是什么，DataLoader 管打乱分批拼盘；
> 长短不齐自己拼，补齐加 mask；
> worker 数 ≥ 读一批 ÷ 训一步，pin_memory 配 non_blocking；多卡 DistributedSampler，每轮 set_epoch。